##### Neural Network

In [ ]:
# Check if cuda is available - ran on Google Colab originally using A100 GPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

###### Define Dataset and Neural Network class

Custom Dataset to use with Training/Testing DataLoader classes when actually training the model.

HousePriceNN is just a simple feedforwards Neural Net with a single output for the predicted house value (since it is not a classification problem).

In [ ]:
class ZillowDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Define the Regression Model
class HousePriceNN(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

###### Hyperparameter, feature testing, and 5-fold cross validation

Testing a small grid search on possible epoch training and learning rate values.

This could be much further extended, especially for learning rate but for the sake of this project was kept pretty simple to explore some values.

Additionally, 5-folds cross validation (using the built in sklearn KFold library) was chosen just simply based on the heuristic that 5 or 10 are generally good working values and 10 would add more time to training.

This exact same training loop is used for both the models that are optimized for MSE and MAE below

###### Fit best model using MSE as Criterion

Since we used both MSE and MAE as our evaluation metrics in previous models, we also want to try fitting the same neural network model using both metrics as our loss function to optimize for.

First up is a model that uses MSE as the loss function for training.

In [ ]:
epochs_grid = [10, 20, 30, 40, 50]
lr_grid = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
batch_size = 32
k_folds = 5

kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

best_avg_loss = float('inf')
best_params = {}
grid_results = []

print("Starting Grid Search...")

# Define the single feature set for the Neural Network, using the already scaled training features
X_nn_train = X_train_sc
y_nn_train = y_train # y_train is already log-transformed

# Determine input features count for the NN model
input_features_count = X_nn_train.shape[1]

for lr, epochs in itertools.product(lr_grid, epochs_grid):
    fold_losses = []
    print(f"\nTesting lr={lr}, epochs={epochs}...")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_nn_train)):
        X_fold_train, y_fold_train = X_nn_train[train_idx], y_nn_train.iloc[train_idx]
        X_fold_val, y_fold_val = X_nn_train[val_idx], y_nn_train.iloc[val_idx]

        train_dataset = ZillowDataset(X_fold_train, y_fold_train.values.reshape(-1, 1))
        val_dataset = ZillowDataset(X_fold_val, y_fold_val.values.reshape(-1, 1))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = HousePriceNN(input_features=input_features_count).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model.train()
        for epoch in range(epochs):
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(batch_X), batch_y)
                loss.backward()
                optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                val_loss += criterion(model(batch_X), batch_y).item() * batch_X.size(0)

        fold_losses.append(val_loss / len(val_loader.dataset))

    avg_loss = sum(fold_losses) / k_folds

    print(f"Fold Losses: {fold_losses}")
    print(f"Average Loss: {avg_loss:.4f}")
    grid_results.append({'lr': lr, 'epochs': epochs, 'val_loss': avg_loss})

    if avg_loss < best_avg_loss:
        best_avg_loss = avg_loss
        best_params = {'lr': lr, 'epochs': epochs}

print(f"\nBest Params: lr={best_params['lr']}, epochs={best_params['epochs']} (Loss: {best_avg_loss:.4f})")

In [ ]:
# Check best hyperparam combos for MSE model
df_results = pd.DataFrame(grid_results)
df_results.sort_values('val_loss').head(10)

Based on the grid search, a learning rate of 0.01 and 50 epochs seemed to do the best on average with the 5-fold cross validation. These are the values we will use for the final testing set, while still acknowledging that due to the simple nature of this model and the lack of deep exploration into all feature combinations, better performing combinations will almost surely exist.

Next, we fit a model using the best hyperparameters from the grid search that used MSE as the loss function.

In [ ]:
# Get the best configuration
best_epochs = best_params['epochs']
best_lr = best_params['lr']

final_train_dataset = ZillowDataset(X_train_sc, y_train.values.reshape(-1, 1))
test_dataset = ZillowDataset(X_test_sc, y_test.values.reshape(-1, 1))

final_train_loader = DataLoader(final_train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

final_model = HousePriceNN(input_features=input_features_count).to(device)
optimizer = optim.Adam(final_model.parameters(), lr=best_lr)
criterion = nn.MSELoss()

train_losses = []
test_losses = []

print(f"Training final model with lr={best_lr}, epochs={best_epochs}...")
for epoch in range(best_epochs):
    # Training
    final_model.train()
    epoch_train_loss = 0
    for batch_X, batch_y in final_train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        preds = final_model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * batch_X.size(0)

    train_losses.append(epoch_train_loss / len(final_train_dataset))

    # Testing per epoch
    final_model.eval()
    epoch_test_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            preds = final_model(batch_X)
            loss = criterion(preds, batch_y)
            epoch_test_loss += loss.item() * batch_X.size(0)

    test_losses.append(epoch_test_loss / len(test_dataset))


# Final RMSE calculation
final_model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        all_preds.append(final_model(batch_X).cpu().numpy())
        all_targets.append(batch_y.cpu().numpy())

# Convert log-transformed predictions and targets back to dollar values
preds_dollars = np.exp(np.vstack(all_preds))
targets_dollars = np.exp(np.vstack(all_targets))
rmse_dollars = np.sqrt(mean_squared_error(targets_dollars, preds_dollars))
print(f"\nFinal Test RMSE in Dollars: ${rmse_dollars:,.2f}")

# Save the best MAE model weights
torch.save(final_model_mae.state_dict(), 'best_zillow_model_mse.pth')
print("MAE Model saved to 'best_zillow_model_mse.pth'")

###### Fit best model using MAE as Criterion

For comparison, next we will fit a model using the same grid search setup with the only difference being MAE is used as the Criterion for optimizing.

In [ ]:
# Grid search for MAE optimized model
epochs_grid = [10, 20, 30, 40, 50]
lr_grid = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
batch_size = 32
k_folds = 5

kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

best_avg_loss_mae = float('inf')
best_params_mae = {}
grid_results_mae = []

print("Starting Grid Search for MAE model...")

for lr, epochs in itertools.product(lr_grid, epochs_grid):
    fold_losses_mae = []
    print(f"\nTesting lr={lr}, epochs={epochs}...")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_nn_train)):
        X_fold_train, y_fold_train = X_nn_train[train_idx], y_nn_train.iloc[train_idx]
        X_fold_val, y_fold_val = X_nn_train[val_idx], y_nn_train.iloc[val_idx]

        train_dataset = ZillowDataset(X_fold_train, y_fold_train.values.reshape(-1, 1))
        val_dataset = ZillowDataset(X_fold_val, y_fold_val.values.reshape(-1, 1))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model_mae = HousePriceNN(input_features=input_features_count).to(device)
        criterion_mae = nn.L1Loss() # L1Loss = MAE in PyTorch
        optimizer_mae = optim.Adam(model_mae.parameters(), lr=lr)

        model_mae.train()
        for epoch in range(epochs):
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer_mae.zero_grad()
                loss = criterion_mae(model_mae(batch_X), batch_y)
                loss.backward()
                optimizer_mae.step()

        model_mae.eval()
        val_loss_mae = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                val_loss_mae += criterion_mae(model_mae(batch_X), batch_y).item() * batch_X.size(0)

        fold_losses_mae.append(val_loss_mae / len(val_loader.dataset))

    avg_loss_mae = sum(fold_losses_mae) / k_folds

    print(f"Fold Losses MAE: {fold_losses_mae}")
    print(f"Average Loss MAE: {avg_loss_mae:.4f}")
    grid_results_mae.append({'lr': lr, 'epochs': epochs, 'val_loss_mae': avg_loss_mae})

    if avg_loss_mae < best_avg_loss_mae:
        best_avg_loss_mae = avg_loss_mae
        best_params_mae = {'lr': lr, 'epochs': epochs}

print(f"\nBest Params MAE: lr={best_params_mae['lr']}, epochs={best_params_mae['epochs']} (Loss: {best_avg_loss_mae:.4f})")

In [ ]:
# Check best hyperparam combos for MAE model
df_maeresults = pd.DataFrame(grid_results_mae)
df_maeresults.sort_values('val_loss_mae').head(10)

After doing grid search on the MAE model, the best parameter combination appears to be lr = .15 with 20 epochs, not too different from the MSE model which had the same best lr but 50 epochs instead.

Furthermore, the losses for 20 and 30 epochs with lr = .15 are quite close together, indicating a pretty trivial difference between one or the other in this case, so we will just go with the best.

In [ ]:
# Get the best configuration for MAE model
best_epochs_mae = best_params_mae['epochs']
best_lr_mae = best_params_mae['lr']
input_features_count_mae = best_params_mae['input_features_count']

final_train_dataset_mae = ZillowDataset(X_train_sc, y_train.values.reshape(-1, 1))
test_dataset_mae = ZillowDataset(X_test_sc, y_test.values.reshape(-1, 1))

final_train_loader_mae = DataLoader(final_train_dataset_mae, batch_size=batch_size, shuffle=True)
test_loader_mae = DataLoader(test_dataset_mae, batch_size=batch_size, shuffle=False)

final_model_mae = HousePriceNN(input_features=input_features_count_mae).to(device)
optimizer_mae = optim.Adam(final_model_mae.parameters(), lr=best_lr_mae)
criterion_mae = nn.L1Loss() # Use MAE Loss for final model

train_losses_mae = []
test_losses_mae = []

print(f"Training final MAE model with lr={best_lr_mae}, epochs={best_epochs_mae}...")
for epoch in range(best_epochs_mae):
    # Training
    final_model_mae.train()
    epoch_train_loss_mae = 0
    for batch_X, batch_y in final_train_loader_mae:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer_mae.zero_grad()
        preds_mae = final_model_mae(batch_X)
        loss_mae = criterion_mae(preds_mae, batch_y)
        loss_mae.backward()
        optimizer_mae.step()
        epoch_train_loss_mae += loss_mae.item() * batch_X.size(0)

    train_losses_mae.append(epoch_train_loss_mae / len(final_train_dataset_mae))

    # Testing per epoch
    final_model_mae.eval()
    epoch_test_loss_mae = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader_mae:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            preds_mae = final_model_mae(batch_X)
            loss_mae = criterion_mae(preds_mae, batch_y)
            epoch_test_loss_mae += loss_mae.item() * batch_X.size(0)

    test_losses_mae.append(epoch_test_loss_mae / len(test_dataset_mae))


# Final RMSE calculation for MAE model
final_model_mae.eval()
all_preds_mae = []
all_targets_mae = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_mae:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        all_preds_mae.append(final_model_mae(batch_X).cpu().numpy())
        all_targets_mae.append(batch_y.cpu().numpy())

# Convert log-transformed predictions and targets back to dollar values using np.exp
preds_dollars_mae = np.exp(np.vstack(all_preds_mae))
targets_dollars_mae = np.exp(np.vstack(all_targets_mae))
rmse_dollars_mae = np.sqrt(mean_squared_error(targets_dollars_mae, preds_dollars_mae))
print(f"\nFinal Test RMSE in Dollars (MAE Model): ${rmse_dollars_mae:,.2f}")

# Save the best MAE model weights
torch.save(final_model_mae.state_dict(), 'best_zillow_model_mae.pth')
print("MAE Model saved to 'best_zillow_model_mae.pth'")

###### MSE Model: Train/Test Loss Curves

In [ ]:
# Plot train/test loss over epochs for the MSE model
plt.figure(figsize=(10, 6))

plt.plot(range(1, best_epochs + 1), train_losses, label='MSE Train Loss', color='orange', linestyle='-')
plt.plot(range(1, best_epochs + 1), test_losses, label='MSE Test Loss', color='blue', linestyle='-')

plt.xlabel('Epochs')
plt.ylabel('Loss (Scaled)')
plt.title('MSE Model: Train/Test Loss Curves')
plt.legend()
plt.grid(True)
plt.show()

###### MAE Model: Train/Test Loss Curves

In [ ]:
# Plot train/test loss over epochs for the MAE model
plt.figure(figsize=(10, 6))

plt.plot(range(1, best_epochs_mae + 1), train_losses_mae, label='MAE Train Loss', color='orange', linestyle='-')
plt.plot(range(1, best_epochs_mae + 1), test_losses_mae, label='MAE Test Loss', color='blue', linestyle='-')

plt.xlabel('Epochs')
plt.ylabel('Loss (Scaled)')
plt.title('MAE Model: Train/Test Loss Curves')
plt.legend()
plt.grid(True)
plt.show()

Both models are able to reach a stable training and test loss after about 10 epochs, from there the gains are extremely marginal, especially for the MSE model. It really did not need 50 epochs; even though that was the "best" performing model in the grid-search, it was trivially different from one that could be trained with 10 epochs.

The MSE model had a final RMSE on the full test data set of 1,286,591.82 while the MAE model had RMSE of 1,242,502.11. So with this metric, we do see slightly better overall predictions from the model that was optimized using the MAE/L1 loss function, with an improvement of about $40,000 in terms of RMSE.

###### Making final predictions on the complete dataset

Since the MAE model performed slightly better, we can use it now to make predictions on the whole dataset and see how they look in terms of dollars against the actual values.

In [ ]:
# Save version of full dataset with only the features used in the final model.
# Use the original X (before train-test split, after one-hot encoding) and scale it using the previously fitted scaler
X_full = X # This 'X' is the one created in cell 14abe4f1, after get_dummies but before train-test split
X_full_scaled = scaler.transform(X_full) # 'scaler' was fit on X_train_sc in cell 7e4512dc

# Convert to PyTorch tensor
X_full_tensor = torch.tensor(X_full_scaled, dtype=torch.float32).to(device)

# Predict prices for all houses in the data
final_model_mae.eval()
with torch.no_grad():
    preds_full_log_scaled = final_model_mae(X_full_tensor).cpu().numpy()

# Undo log transformation to get actual dollar values.
preds_full_dollars = np.exp(preds_full_log_scaled)

# Create comparison for actual vs. predicted price
df_comparison = pd.DataFrame({
    'Actual Price': df_filtered['price'].values, # Use the original df_filtered price
    'Predicted Price': preds_full_dollars.flatten()
})
df_comparison['Difference'] = df_comparison['Actual Price'] - df_comparison['Predicted Price']

display(df_comparison.head(15))